In [4]:
import os
import pandas as pd
import tkinter as tk
from tkinter import filedialog
from mcap.reader import make_reader
from mcap_ros2.decoder import Decoder
from mcap.writer import Writer



import os
import mcap
from mcap.reader import make_reader
from mcap.writer import Writer
import tkinter as tk
from tkinter import filedialog


class Merge:

    def merge_mcap_files(self, input_folder, output_file):
        # 1. Obtener y ordenar los ficheros .mcap de la carpeta
        bag_files = sorted([
            os.path.join(input_folder, f) 
            for f in os.listdir(input_folder) 
            if f.endswith(".mcap")
        ])

        if not bag_files:
            print(f"No se encontraron archivos .mcap en {input_folder}")
            return

        print(f"Fusionando {len(bag_files)} archivos...")

        with open(output_file, "wb") as f_out:
            writer = Writer(f_out)
            # Mantener el perfil "ros2" permite que el archivo resultante 
            # sea reconocido correctamente por herramientas como Foxglove.
            writer.start(profile="ros2") 

            # Diccionarios para evitar duplicar esquemas y canales
            # Usamos los nombres/tópicos como claves para consolidar
            schema_map = {}  # nombre_esquema -> schema_id
            channel_map = {} # nombre_topico -> channel_id

            for bag_path in bag_files:
                print(f"Procesando: {os.path.basename(bag_path)}")

                with open(bag_path, "rb") as f_in:
                    reader = make_reader(f_in)

                    # iter_messages() en la librería base devuelve (schema, channel, message)
                    # sin necesidad de tener las bibliotecas de ROS 2 instaladas.
                    for schema, channel, message in reader.iter_messages():

                        # A. Registrar el esquema si es la primera vez que lo vemos
                        if schema.name not in schema_map:
                            schema_id = writer.register_schema(
                                name=schema.name,
                                encoding=schema.encoding,
                                data=schema.data
                            )
                            schema_map[schema.name] = schema_id

                        # B. Registrar el canal (tópico) si es la primera vez
                        if channel.topic not in channel_map:
                            c_id = writer.register_channel(
                                schema_id=schema_map[schema.name],
                                topic=channel.topic,
                                message_encoding=channel.message_encoding,
                                metadata=channel.metadata
                            )
                            channel_map[channel.topic] = c_id

                        # C. Escribir el mensaje (copia directa de bytes)
                        writer.add_message(
                            channel_id=channel_map[channel.topic],
                            log_time=message.log_time,
                            data=message.data,
                            publish_time=message.publish_time,
                            sequence=message.sequence
                        )

            writer.finish()
        print(f"\nÉxito. Archivo fusionado guardado en: {output_file}")



class ROS2BagMCAPToCSV:
    def __init__(self):
        self.decoder = Decoder()


    @staticmethod
    def flatten_dict(d, parent_key='', sep='.'):
        items = []

        # 1. Intentar convertir a diccionario de todas las formas posibles
        if hasattr(d, '__getstate__'):
            data_map = d.__getstate__()
        elif hasattr(d, 'get_fields_and_field_types'): # Específico de algunas versiones de ROS2
            data_map = {field: getattr(d, field) for field in d.get_fields_and_field_types().keys()}
        elif hasattr(d, '__dict__'):
            data_map = vars(d)
        elif isinstance(d, dict):
            data_map = d
        else:
            # Si llegamos aquí y es un objeto, intentamos ver si tiene un .data (común en String.msg)
            if hasattr(d, 'data'):
                return {parent_key: str(d.data)}
            return {parent_key: str(d)}

        for k, v in data_map.items():
            if k.startswith('_'): continue

            new_key = f"{parent_key}{sep}{k}" if parent_key else k

            # 2. Análisis del valor 'v'
            # Si es un objeto complejo (pero no un string/número/bytes)
            if hasattr(v, '__dict__') or hasattr(v, '__getstate__') or isinstance(v, dict):
                # Caso especial: Si el objeto tiene un .data que es el valor real
                if hasattr(v, 'data') and not (hasattr(v.data, '__dict__') or isinstance(v.data, dict)):
                    val = v.data
                    if isinstance(val, bytes): val = val.decode('utf-8', errors='ignore')
                    items.append((new_key, val))
                else:
                    items.extend(ROS2BagMCAPToCSV.flatten_dict(v, new_key, sep=sep).items())

            # 3. Tratamiento de tipos básicos
            elif isinstance(v, bytes):
                items.append((new_key, v.decode('utf-8', errors='ignore')))
            elif isinstance(v, list) or isinstance(v, tuple):
                # Si la lista contiene bytes, decodificarlos
                clean_list = [x.decode('utf-8', errors='ignore') if isinstance(x, bytes) else x for x in v]
                items.append((new_key, str(clean_list)))
            elif v is None:
                items.append((new_key, ""))
            else:
                # Forzamos que sea un tipo básico de Python
                items.append((new_key, v))

        return dict(items)


    def process_single_file(self, mcap_path):
        output_dir = os.path.splitext(mcap_path)[0] + "_csv"
        os.makedirs(output_dir, exist_ok=True)
        
        data_by_topic = {}
        processed_count = 0
        
        print(f"📂 Abriendo: {os.path.basename(mcap_path)}")
        
        try:
            with open(mcap_path, "rb") as f:
                reader = make_reader(f)
                
                # Usamos un try-except DENTRO del bucle para salvar lo procesado hasta el error
                try:
                    for schema, channel, message in reader.iter_messages():
                        try:
                            topic_name = channel.topic
                            ros_msg = self.decoder.decode(schema, message)

                            msg_dict = {
                                slot: getattr(ros_msg, slot) 
                                for slot in dir(ros_msg) 
                                if not slot.startswith('_') and not callable(getattr(ros_msg, slot))
                            }

                            record = {
                                "log_time_s": message.log_time / 1e9,
                                "publish_time_s": message.publish_time / 1e9,
                                **self.flatten_dict(msg_dict)
                            }

                            if topic_name not in data_by_topic:
                                data_by_topic[topic_name] = []
                            data_by_topic[topic_name].append(record)
                            processed_count += 1
                            
                        except Exception as msg_err:
                            # Si falla un mensaje individual (decodificación), saltamos al siguiente
                            continue

                except Exception as stream_err:
                    # Este es el error "unpack_from" (corrupción de buffer)
                    print(f"⚠️ Corrupción detectada en el stream. Se recuperaron {processed_count} mensajes.")
                    print(f"Detalle del error: {stream_err}")

            # EXPORTACIÓN (Se ejecuta aunque el bucle de arriba haya fallado a mitad)
            if not data_by_topic:
                print(f"❌ No se pudieron recuperar datos de {os.path.basename(mcap_path)}")
                return

            for topic_name, records in data_by_topic.items():
                df = pd.DataFrame(records)
                clean_name = topic_name.strip("/").replace("/", "_")
                output_file = os.path.join(output_dir, f"{clean_name}.csv")
                
                df.to_csv(
                    output_file, 
                    index=False,
                    quoting=1,
                    decimal='.',
                    escapechar='\\',
                    encoding='utf-8-sig'
                )
            
            print(f"✅ Datos recuperados guardados en: {output_dir}")

        except Exception as e:
            print(f"❌ Error crítico en el archivo {mcap_path}: {e}")

    def process_single_file_old(self, mcap_path):
        output_dir = os.path.splitext(mcap_path)[0] + "_csv"
        os.makedirs(output_dir, exist_ok=True)
        
        data_by_topic = {}
        
        try:
            with open(mcap_path, "rb") as f:
                reader = make_reader(f)
                for schema, channel, message in reader.iter_messages():
                    topic_name = channel.topic
                    ros_msg = self.decoder.decode(schema, message)
                    print(channel.topic)
                    print(type(ros_msg))
                    print(ros_msg)
                    # Filtrar manualmente solo los campos de datos útiles
                    #msg_dict = {
                    #    slot: getattr(ros_msg, slot) 
                    #    for slot in dir(ros_msg) 
                    #    if not slot.startswith('_') and not callable(getattr(ros_msg, slot))
                    #}

                    record = {
                        "log_time_s": message.log_time / 1e9,
                        "publish_time_s": message.publish_time / 1e9,
                        **self.flatten_dict(msg_dict)
                    }

                    if topic_name not in data_by_topic:
                        data_by_topic[topic_name] = []
                    data_by_topic[topic_name].append(record)

            for topic_name, records in data_by_topic.items():
                if not records: continue
                
                df = pd.DataFrame(records)
                clean_name = topic_name.strip("/").replace("/", "_")
                
                # Exportación robusta
                df.to_csv(
                    os.path.join(output_dir, f"{clean_name}.csv"), 
                    index=False,
                    quoting=1,       # Pone comillas a todos los strings (evita errores de lectura)
                    decimal='.',     # Fuerza el punto decimal para floats
                    escapechar='\\',
                    #quotechar ='"',
                    encoding='utf-8-sig' # Asegura que los textos no se corrompan
                )
            
            print(f"✅ Procesado con éxito: {os.path.basename(mcap_path)}")
        except Exception as e:
            print(f"❌ Error procesando {mcap_path}: {e}")
            
    def run_recursive(self, root_folder):
        """Busca y procesa todos los .mcap en la carpeta y subcarpetas."""
        mcap_files = []
        for root, dirs, files in os.walk(root_folder):
            for file in files:
                if file.endswith(".mcap"):
                    mcap_files.append(os.path.join(root, file))
        
        if not mcap_files:
            print("No se encontraron archivos .mcap en el directorio seleccionado.")
            return

        print(f"🔍 Se encontraron {len(mcap_files)} archivos. Iniciando conversión...")
        for path in mcap_files:
            print(f"🚀 Procesando: {path}")
            self.process_single_file(path)
        print("\n✨ ¡Proceso finalizado!")

        
        
        
        
def main():
    root = tk.Tk()
    root.withdraw()
    root.attributes("-topmost", True)

    folder_path = filedialog.askdirectory(title="Selecciona la carpeta raíz para buscar MCAPs")

    if folder_path:
        converter = ROS2BagMCAPToCSV()
        #m=Merge()
        #output_path = os.path.join(folder_path, "merged_session.mcap")
        #m.merge_mcap_files(folder_path, output_path)
        converter.run_recursive(folder_path)
    else:
        print("Operación cancelada.")

if __name__ == '__main__':
    main()

🔍 Se encontraron 1 archivos. Iniciando conversión...
🚀 Procesando: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag/USUARIO1_0_2_2026_06_15-09_15_59\USUARIO1_0_2_2026_06_15-09_15_59_0.mcap
📂 Abriendo: USUARIO1_0_2_2026_06_15-09_15_59_0.mcap
✅ Datos recuperados guardados en: C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag/USUARIO1_0_2_2026_06_15-09_15_59\USUARIO1_0_2_2026_06_15-09_15_59_0_csv

✨ ¡Proceso finalizado!


In [13]:
import sys
import os

print("--- COMPROBACIÓN ---")
print("Python ejecutable desde aquí:", sys.executable)
print("Carpeta donde Python está buscando:", os.getcwd())
print("--------------------")

--- COMPROBACIÓN ---
Python ejecutable desde aquí: C:\Users\alber\anaconda3\python.exe
Carpeta donde Python está buscando: C:\Users\alber\Documents\GitHub\SillaSamu\Postprocessing
--------------------


In [14]:
name = 'C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag/USUARIO1_0_2_2026_06_15-09_15_59'
posicion = name.rfind('/')
subcadena = name[posicion+1:]
posicion = subcadena.find('_')
sujeto = subcadena[:posicion]
subcadena = subcadena[posicion+1:]
posicion = subcadena.find('_')
exp = int(subcadena[:posicion])
subcadena = subcadena[posicion+1:]
posicion = subcadena.find('_')
modo = int(subcadena[:posicion])
subcadena = subcadena[posicion+1:]

posicion = subcadena.find('-')
fecha = (subcadena[:posicion])

hora = subcadena[posicion+1:]


print(f'Sujeto:{sujeto},Exp:{exp},Modo:{modo},Fecha:{fecha}, Hora:{hora}')

Sujeto:USUARIO1,Exp:0,Modo:2,Fecha:2026_06_15, Hora:09_15_59


In [15]:
print(fecha.replace('_',''))

20260615


In [55]:
from pathlib import Path
import pandas as pd
import shutil

nombre_directorio = "C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/logger/"
nombre_directorio_salida = "C:/Users/alber/Documents/GitHub/SillaSamu/Postprocessing/samuchair_bag/PRUEBA/"
# Definimos la ruta de la carpeta
carpeta = Path(nombre_directorio_salida)

# 1. Si existe, la borramos con todo su contenido
if carpeta.exists() and carpeta.is_dir():
    shutil.rmtree(carpeta)
    print(f"Carpeta {carpeta.name} borrada con éxito.")

# 2. La creamos de nuevo (vacía)
carpeta.mkdir(parents=True, exist_ok=True)
print(f"Carpeta {carpeta.name} creada de nuevo desde cero.")

# 1. Define la carpeta donde quieres buscar
carpeta_base = Path(nombre_directorio)
subcadena = fecha.replace('_','')

# 2. Buscar RECURSIVAMENTE (En la carpeta y todas sus subcarpetas)
# El '**/*' le dice que entre a todo, y '*subcadena*' busca el texto
archivos_encontrados = list(carpeta_base.glob(f"**/*{subcadena}*"))
df = pd.DataFrame()
topics_name =set()
# Mostrar los resultados
for archivo in archivos_encontrados:
    print('NOMBRE DE ARCHIVO')
    print(archivo)  # Imprime la ruta completa limpia
    df = pd.read_csv(archivo,sep=';',encoding='utf-8-sig')
    topics_name.update(set(df['topic']))
    
    
    for top in topics_name:
        print('-----------------')
        print(top)
        data = df[ df['topic']==top]
        #print(data['topic'])
        archivo = nombre_directorio_salida+str(top)+'.csv'
        data.to_csv(
            archivo, 
            mode='a', 
            index=False, 
            header=False,
            sep=';', 
            encoding='utf-8-sig'
        )
        res= data.index[data['topic']=='directorio_grabacion']
        
        print(res)
        if len(res)>0:
            print(data.iloc[res,2])
            print('..................')
            if "USUARIO1_0_2" in str(data.iloc[res,2]):
                print('************ENCONTRADO**************')
                print
        
print(topics_name)

Carpeta PRUEBA borrada con éxito.
Carpeta PRUEBA creada de nuevo desde cero.
NOMBRE DE ARCHIVO
C:\Users\alber\Documents\GitHub\SillaSamu\Postprocessing\logger\log_20260615_091530.csv
-----------------
pot_esp32
Int64Index([], dtype='int64')
-----------------
left_wheel_steps
Int64Index([], dtype='int64')
-----------------
tactil_der
Int64Index([], dtype='int64')
-----------------
op_manual_asistida
Int64Index([], dtype='int64')
-----------------
right_wheel_steps
Int64Index([], dtype='int64')
-----------------
Ejecucion
Int64Index([], dtype='int64')
-----------------
control_grabacion
Int64Index([], dtype='int64')
-----------------
modo_exp
Int64Index([], dtype='int64')
-----------------
name_movil
Int64Index([], dtype='int64')
-----------------
directorio_grabacion
Int64Index([62], dtype='int64')


IndexError: positional indexers are out-of-bounds

In [60]:
print ('directorio_grabacion' in topics_name)

True
